# TripGenie: Hotel Ranking Exploration

This notebook explores the hotel dataset and demonstrates the ML-based hotel ranking system.

**Sections:**
1. Dataset exploration
2. Feature engineering walkthrough
3. Ranking demonstration for different trip intents
4. Feature weight sensitivity analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Set a clean style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

from app.schemas.domain import BudgetLevel, TravelStyle, TripIntent
from app.ml.features import extract_features, FEATURE_FUNCTIONS
from app.ml.ranker import HotelRanker
from app.services.dataset_service import DatasetService

print('Imports successful.')

## 1. Dataset Exploration

In [ ]:
# Load hotel data via the dataset service
dataset = DatasetService()
hotels = dataset.get_hotels()
print(f'Total hotels loaded: {len(hotels)}')
print(f'Cities: {sorted({h.city for h in hotels})}')

# Convert to DataFrame for exploration
df = pd.DataFrame([
    {
        'hotel_id': h.hotel_id,
        'city': h.city,
        'name': h.name,
        'price_level': h.price_level,
        'avg_review_score': h.avg_review_score,
        'location_score': h.location_score,
        'near_museum': int(h.near_museum),
        'near_nightlife': int(h.near_nightlife),
        'near_public_transport': int(h.near_public_transport),
        'romantic': int(h.romantic),
        'luxury': int(h.luxury),
        'budget': int(h.budget),
        'family_friendly': int(h.family_friendly),
        'business_friendly': int(h.business_friendly),
    }
    for h in hotels
])

df.head()

In [ ]:
print('Dataset summary:')
print(df[['price_level', 'avg_review_score', 'location_score']].describe().round(2))

In [ ]:
# Hotel count by city and price level
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hotels by city
city_counts = df['city'].value_counts()
axes[0].bar(city_counts.index, city_counts.values, color=['#2196F3', '#4CAF50', '#FF9800', '#E91E63', '#9C27B0'])
axes[0].set_title('Hotels by City')
axes[0].set_xlabel('City')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Price level distribution
price_labels = {1: 'Budget', 2: 'Mid-range', 3: 'Upper mid', 4: 'Luxury'}
price_counts = df['price_level'].map(price_labels).value_counts()
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']
axes[1].pie(price_counts.values, labels=price_counts.index, autopct='%1.0f%%',
            colors=colors, startangle=90)
axes[1].set_title('Price Level Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Review score distribution by city
fig, ax = plt.subplots(figsize=(12, 5))
for city in sorted(df['city'].unique()):
    city_df = df[df['city'] == city]
    ax.scatter(city_df['location_score'], city_df['avg_review_score'],
               label=city, s=80, alpha=0.8)

ax.set_xlabel('Location Score')
ax.set_ylabel('Average Review Score')
ax.set_title('Location vs. Review Scores by City')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Feature prevalence across all hotels
feature_cols = ['near_museum', 'near_nightlife', 'near_public_transport', 
                'romantic', 'luxury', 'budget', 'family_friendly', 'business_friendly']

feature_pct = df[feature_cols].mean() * 100

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh([f.replace('_', ' ').title() for f in feature_pct.index],
               feature_pct.values, color='#2196F3', alpha=0.85)
ax.set_xlabel('% of Hotels with Feature')
ax.set_title('Hotel Feature Prevalence')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
for bar, val in zip(bars, feature_pct.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0f}%', va='center', fontsize=10)
plt.tight_layout()
plt.show()

## 2. Feature Engineering Walkthrough

In [ ]:
# Define a sample trip intent for Amsterdam
cultural_intent = TripIntent(
    city='Amsterdam',
    days=4,
    travelers=2,
    budget_level=BudgetLevel.MID,
    interests=['museums', 'canals', 'culture'],
    travel_style=TravelStyle.CULTURAL,
    accommodation_preferences=['near public transport', 'central location'],
)

print('Trip intent summary:', cultural_intent.to_summary())

In [ ]:
# Compute features for all Amsterdam hotels
amsterdam_hotels = [h for h in hotels if h.city == 'Amsterdam']

feature_data = []
for hotel in amsterdam_hotels:
    fv = extract_features(hotel, cultural_intent)
    row = {'hotel': hotel.name, **fv.to_dict()}
    feature_data.append(row)

feature_df = pd.DataFrame(feature_data).set_index('hotel')
feature_df.round(3)

In [ ]:
# Visualise feature matrix as a heatmap
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(feature_df.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)

ax.set_xticks(range(len(feature_df.columns)))
ax.set_xticklabels([c.replace('_', ' ').title() for c in feature_df.columns],
                   rotation=35, ha='right', fontsize=10)
ax.set_yticks(range(len(feature_df.index)))
ax.set_yticklabels(feature_df.index, fontsize=9)
ax.set_title('Feature Matrix: Amsterdam Hotels vs. Cultural Trip Intent')
plt.colorbar(im, ax=ax, label='Feature Score (0-1)')
plt.tight_layout()
plt.show()

## 3. Ranking Demonstration

In [ ]:
# Rank Amsterdam hotels for the cultural intent
ranker = HotelRanker()
ranked = ranker.rank(amsterdam_hotels, cultural_intent, top_k=12)

print(f'Top 5 hotels for: {cultural_intent.to_summary()}\n')
print(f'{"Rank":<5} {"Hotel":<35} {"Score":<8} {"Price":<10} {"Review":<8} {"Location":<9}')
print('-' * 80)
for rh in ranked[:5]:
    h = rh.hotel
    print(f'#{rh.rank:<4} {h.name:<35} {rh.score:.3f}    {h.price_level:<10} {h.avg_review_score:<8.1f} {h.location_score:.1f}')

In [ ]:
# Compare rankings for different trip intents
intents = {
    'Cultural (mid budget)': TripIntent(
        city='Amsterdam', days=4, travelers=2,
        budget_level=BudgetLevel.MID,
        interests=['museums', 'culture'],
        travel_style=TravelStyle.CULTURAL,
    ),
    'Romantic (luxury)': TripIntent(
        city='Amsterdam', days=4, travelers=2,
        budget_level=BudgetLevel.LUXURY,
        interests=['romantic', 'luxury'],
        travel_style=TravelStyle.ROMANTIC,
    ),
    'Budget solo': TripIntent(
        city='Amsterdam', days=3, travelers=1,
        budget_level=BudgetLevel.BUDGET,
        interests=['sightseeing'],
        travel_style=TravelStyle.CULTURAL,
    ),
    'Family': TripIntent(
        city='Amsterdam', days=5, travelers=4,
        budget_level=BudgetLevel.MID,
        interests=['family', 'parks'],
        travel_style=TravelStyle.FAMILY,
    ),
}

# Collect scores for top hotel per intent
results = {}
for intent_name, intent in intents.items():
    ranked_for_intent = ranker.rank(amsterdam_hotels, intent, top_k=5)
    results[intent_name] = {
        'top_hotel': ranked_for_intent[0].hotel.name,
        'top_score': ranked_for_intent[0].score,
        'ranking': [r.hotel.name for r in ranked_for_intent[:3]],
    }
    print(f'Intent: {intent_name}')
    print(f'  Top hotel: {results[intent_name]["top_hotel"]} (score: {results[intent_name]["top_score"]:.3f})')
    print(f'  Top 3: {results[intent_name]["ranking"]}\n')

In [ ]:
# Visualise the top 5 hotel scores for each intent
fig, axes = plt.subplots(1, len(intents), figsize=(16, 5), sharey=False)

colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']
for ax, (intent_name, intent), color in zip(axes, intents.items(), colors):
    ranked_for_intent = ranker.rank(amsterdam_hotels, intent, top_k=5)
    names = [r.hotel.name.replace(' ', '\n') for r in ranked_for_intent]
    scores = [r.score for r in ranked_for_intent]
    bars = ax.bar(range(len(names)), scores, color=color, alpha=0.8)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, fontsize=7)
    ax.set_ylim(0, 1.0)
    ax.set_title(intent_name, fontsize=10)
    ax.set_ylabel('Score' if ax == axes[0] else '')
    for bar, score in zip(bars, scores):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{score:.2f}', ha='center', fontsize=8)

plt.suptitle('Top 5 Hotel Rankings by Trip Intent', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Feature Weight Sensitivity Analysis

In [ ]:
# Show how the top-ranked hotel changes as we vary one key weight
from app.ml.ranker import DEFAULT_WEIGHTS

base_weights = DEFAULT_WEIGHTS.copy()
sensitivity_results = []

for museum_weight in np.linspace(0, 0.5, 11):
    weights = base_weights.copy()
    weights['museum_affinity'] = museum_weight
    
    ranker_test = HotelRanker(feature_weights=weights)
    ranked_test = ranker_test.rank(amsterdam_hotels, cultural_intent, top_k=1)
    sensitivity_results.append({
        'museum_weight': museum_weight,
        'top_hotel': ranked_test[0].hotel.name,
        'top_score': ranked_test[0].score,
    })

sens_df = pd.DataFrame(sensitivity_results)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Score vs weight
ax1.plot(sens_df['museum_weight'], sens_df['top_score'], 'o-', color='#2196F3', linewidth=2)
ax1.set_xlabel('museum_affinity weight')
ax1.set_ylabel('Top hotel score')
ax1.set_title('Score Sensitivity to Museum Weight')

# Top hotel name frequency
hotel_freq = sens_df['top_hotel'].value_counts()
ax2.bar(hotel_freq.index, hotel_freq.values, color='#4CAF50', alpha=0.85)
ax2.set_title('Top Hotel Frequency Across Weight Variations')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

print('Sensitivity analysis complete.')
print(f'As museum_affinity weight increases, ranking is dominated by: {hotel_freq.index[0]}')

In [ ]:
# Feature contribution breakdown for the top hotel
top_ranked = ranker.rank(amsterdam_hotels, cultural_intent, top_k=1)[0]

contribs = {c.feature.replace('_', ' ').title(): c.contribution 
            for c in top_ranked.feature_contributions}

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(contribs.keys(), contribs.values(), 
              color=['#4CAF50' if v > 0.04 else '#2196F3' for v in contribs.values()])
ax.set_title(f'Score Contribution Breakdown\n{top_ranked.hotel.name} (Total: {top_ranked.score:.3f})')
ax.set_ylabel('Score Contribution')
ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, contribs.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()